# This is an example of domain knowledge that can be represented as a Bayesian Network:

I can contact my boss by email or skype. I can get a quick response or not. If she/he is in office (only 50% of time), she/he will respond in 50% cases. The boss usually does not check skype messages when in office, but 50% of the time checks when not in office. The boss usually checks email messages when in office and is unlikely to check it when not in office. The boss very likely responses, if have checked both skype and e-mail. The boss likely responses, if have checked one of skype or email. The boss does not response, if have not checked neither skype nor email.

In this lab, we will see how to implement this example in Python.


In [1]:
#Importing Library

import numpy as np

In [3]:
from pgmpy.factors.discrete import TabularCPD

In [7]:
from pgmpy.models import BayesianNetwork

Pgmpy is a pure python implementation for Bayesian Networks with a focus on modularity and extensibility:
https://pgmpy.org/
The pgmpy website provides great tutorials and examples of BN.

We define the network structure, named office_model, using BayesianModel

In [9]:
office_model = BayesianNetwork([('InOffice', 'Skype'),
                              ('InOffice', 'Email'),
                              ('Skype', 'Response'),
                              ('Email', 'Response')])

TabularCPD defines the conditional probability distribution table (cpd table). We need these tables for each node. After defining them, we add them all to the model.

In [10]:
inoffice_cpd = TabularCPD(
    variable = 'InOffice',
    variable_card = 2,   # cardinality
    values = [[0.5], [0.5]])  # ['yes', 'no']
    


In [11]:
skype_cpd = TabularCPD(
    variable = 'Skype',
    variable_card = 2,
    values = [[.1, .5],
              [.9, .5]],
    evidence = ['InOffice'],
    evidence_card = [2])

In [12]:
email_cpd = TabularCPD(
    variable = 'Email',
    variable_card = 2,
    values = [[.8, .2],
              [.2, .8]],
    evidence = ['InOffice'],
    evidence_card = [2])

In [13]:
response_cpd = TabularCPD(
    variable = 'Response',
    variable_card = 2,
    values = [[.99, .90, .90, 0.0],
              [.01, .10, .10, 1.0]],
    evidence = ['Skype', 'Email'],
    evidence_card = [2,2])

In [15]:
office_model.add_cpds(inoffice_cpd, skype_cpd, email_cpd, response_cpd)

it is always good to check if got all CPDs correctly, ehat dependences are, etc.

In [16]:
office_model.get_cpds()

[<TabularCPD representing P(InOffice:2) at 0x21139910e30>,
 <TabularCPD representing P(Skype:2 | InOffice:2) at 0x21166379fd0>,
 <TabularCPD representing P(Email:2 | InOffice:2) at 0x21165e3e7e0>,
 <TabularCPD representing P(Response:2 | Skype:2, Email:2) at 0x21166152270>]

In [17]:
office_model.active_trail_nodes('InOffice')

{'InOffice': {'Email', 'InOffice', 'Response', 'Skype'}}

In [18]:
office_model.local_independencies('InOffice')

In [19]:
office_model.get_independencies()

(InOffice ⟂ Response | Skype, Email)
(Skype ⟂ Email | InOffice)
(Response ⟂ InOffice | Skype, Email)
(Email ⟂ Skype | InOffice)


We import VariableElimination, so we can start probabilistic inference to calculate various the probabilities. For example, what is the probability of receiving a response? what a probability of receiving a response by skype? by email? and so on


In [27]:
from pgmpy.inference import VariableElimination

In [28]:
office_infer = VariableElimination(office_model)

In [29]:
prob_response = office_infer.query(variables = ['Response'], joint=False)

In [30]:
print(prob_response['Response'])

+-------------+-----------------+
| Response    |   phi(Response) |
+=============+=================+
| Response(0) |          0.6471 |
+-------------+-----------------+
| Response(1) |          0.3529 |
+-------------+-----------------+


In [31]:
prob_skype = office_infer.query(variables = ['Skype'], joint=False)

In [32]:
print(prob_skype['Skype'])

+----------+--------------+
| Skype    |   phi(Skype) |
+==========+==============+
| Skype(0) |       0.3000 |
+----------+--------------+
| Skype(1) |       0.7000 |
+----------+--------------+


In [33]:
prob_email = office_infer.query(variables = ['Email'], joint=False)

In [34]:
print(prob_email['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.5000 |
+----------+--------------+
| Email(1) |       0.5000 |
+----------+--------------+


In [35]:
prob_inoffice = office_infer.query(variables = ['InOffice'], joint=False)

In [36]:
print(prob_inoffice['InOffice'])

+-------------+-----------------+
| InOffice    |   phi(InOffice) |
+=============+=================+
| InOffice(0) |          0.5000 |
+-------------+-----------------+
| InOffice(1) |          0.5000 |
+-------------+-----------------+


In [37]:
prob_response_inoffice = office_infer.query(
        variables = ['Response'], joint=False,
        evidence = {'InOffice':0})

In [38]:
print(prob_response_inoffice['Response'])

+-------------+-----------------+
| Response    |   phi(Response) |
+=============+=================+
| Response(0) |          0.7452 |
+-------------+-----------------+
| Response(1) |          0.2548 |
+-------------+-----------------+


In [39]:
prob_email_inoffice = office_infer.query(
        variables = ['Email'], joint=False,
        evidence = {'InOffice':0})

In [40]:
print(prob_email_inoffice['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.8000 |
+----------+--------------+
| Email(1) |       0.2000 |
+----------+--------------+


In [41]:
prob_skype_inoffice = office_infer.query(
        variables = ['Skype'], joint=False,
        evidence = {'InOffice':0})

In [42]:
print(prob_skype_inoffice['Skype'])

+----------+--------------+
| Skype    |   phi(Skype) |
+==========+==============+
| Skype(0) |       0.1000 |
+----------+--------------+
| Skype(1) |       0.9000 |
+----------+--------------+


In [43]:
prob_inoffice_response = office_infer.query(
        variables = ['InOffice'], joint=False,
        evidence = {'Response':0})

In [44]:
print(prob_inoffice_response['InOffice'])

+-------------+-----------------+
| InOffice    |   phi(InOffice) |
+=============+=================+
| InOffice(0) |          0.5758 |
+-------------+-----------------+
| InOffice(1) |          0.4242 |
+-------------+-----------------+


In [45]:
prob_email_response = office_infer.query(
        variables = ['Email'], joint=False,
        evidence = {'Response':0})

In [46]:
print(prob_email_response['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.7079 |
+----------+--------------+
| Email(1) |       0.2921 |
+----------+--------------+


In [47]:
prob_skype_response = office_infer.query(
        variables = ['Skype'], joint=False,
        evidence = {'Response':0})

In [48]:
print(prob_skype_response['Skype'])

+----------+--------------+
| Skype    |   phi(Skype) |
+==========+==============+
| Skype(0) |       0.4298 |
+----------+--------------+
| Skype(1) |       0.5702 |
+----------+--------------+


In [49]:
prob_inoffice_response_skype = office_infer.query(
        variables = ['InOffice'], joint=False,
        evidence = {'Response':0, 'Skype':0})

In [50]:
print(prob_inoffice_response_skype['InOffice'])

+-------------+-----------------+
| InOffice    |   phi(InOffice) |
+=============+=================+
| InOffice(0) |          0.1748 |
+-------------+-----------------+
| InOffice(1) |          0.8252 |
+-------------+-----------------+


In [51]:
prob_response_inoffice_email = office_infer.query(
        variables = ['Email'], joint=False,
        evidence = {'Response':0, 'Skype':0})

In [52]:
print(prob_response_inoffice_email['Email'])

+----------+--------------+
| Email    |   phi(Email) |
+==========+==============+
| Email(0) |       0.3204 |
+----------+--------------+
| Email(1) |       0.6796 |
+----------+--------------+
